In [1]:
pip install sounddevice pyttsx3

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.3/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 3.2 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
import time
import sounddevice as sd
import numpy as np
import librosa
import pandas as pd
import joblib
import pyttsx3

# ==========================================
# 1. INISIALISASI MODEL & SUARA PERINGATAN
# ==========================================
print("Memuat model SVM dan scaler...")
model = joblib.load("../svm_audio_model.pkl")
scaler = joblib.load("../audio_scaler.pkl")
feature_names = joblib.load("../feature_names.pkl")

# Inisialisasi engine suara (Text-to-Speech)
engine = pyttsx3.init()
engine.setProperty('rate', 145)   # Kecepatan bicara (sedikit santai)
engine.setProperty('volume', 1.0) # Volume maksimal (0.0 - 1.0)

# ==========================================
# 2. PARAMETER PEREKAMAN
# ==========================================
SAMPLE_RATE = 22050  # Harus sama persis dengan saat training
DURATION = 1.5       # Interval mendengar audio (tiap 1.5 detik sekali)
NUM_SAMPLES = int(SAMPLE_RATE * DURATION)

# Pengatur jeda agar tidak berisik memperingatkan terus-menerus
JEDA_PERINGATAN = 4.0  # Minimal jeda 4 detik sebelum bersuara lagi
waktu_terakhir_peringatan = 0.0

def extract_features(y, sr):
    """Ekstraksi 16 fitur audio yang sama dengan saat data training."""
    rms = librosa.feature.rms(y=y)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    zcr = librosa.feature.zero_crossing_rate(y=y)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    row = {
        'rms': np.mean(rms),
        'centroid': np.mean(centroid),
        'zcr': np.mean(zcr),
    }

    mfcc_mean = np.mean(mfcc, axis=1)
    for i, val in enumerate(mfcc_mean, start=1):
        row[f'mfcc_{i}'] = val

    return row

# ==========================================
# 3. LOOP PENDENGAR SUARA SEKITAR
# ==========================================
print("\n" + "="*55)
print("SISTEM PENDETEKSI KEBISINGAN RUANGAN SEDANG AKTIF")
print("Mikrofon sedang mendengarkan area sekitar...")
print("Tekan Ctrl+C di terminal untuk mematikan sistem.")
print("="*55 + "\n")

try:
    while True:
        # A. Rekam audio sekitar secara langsung dari mikrofon
        rekaman = sd.rec(
            frames=NUM_SAMPLES,
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype='float32'
        )
        sd.wait() # Tunggu 1.5 detik sampai cuplikan suara selesai diambil

        y = rekaman.flatten()

        # B. Cek ambang batas awal (jika ruangan sangat sunyi, lewati kalkulasi)
        if np.max(np.abs(y)) < 0.02:
            print("[STATUS] Ruangan sangat sunyi...            ", end="\r")
            continue

        # C. Ekstraksi Fitur dari suara yang baru saja terdengar
        fitur_dict = extract_features(y, SAMPLE_RATE)
        df_input = pd.DataFrame([fitur_dict])[feature_names]

        # D. Prediksi dengan Model SVM
        scaled_input = scaler.transform(df_input)
        prediksi = model.predict(scaled_input)[0]
        probabilitas = model.predict_proba(scaled_input)[0]
        skor_kelas = dict(zip(model.classes_, probabilitas))

        waktu_sekarang = time.time()

        # E. Logika Bunyi Peringatan
        if prediksi == "berisik" and skor_kelas["berisik"] >= 0.70:
            persen = skor_kelas["berisik"] * 100
            print(f"\n[⚠️ TERDETEKSI BISING] Keyakinan: {persen:.1f}%")

            # Cegah suara peringatan tumpang-tindih (beri jeda)
            if (waktu_sekarang - waktu_terakhir_peringatan) > JEDA_PERINGATAN:
                print(">>> [SPEAKER BERBUNYI]: 'Tolong jangan berisik!'")
                engine.say("Tolong jangan berisik")
                engine.runAndWait()
                waktu_terakhir_peringatan = time.time()
        else:
            persen_tenang = skor_kelas["non-berisik"] * 100
            print(f"[STATUS] Aman / Kondusif ({persen_tenang:.1f}%)    ", end="\r")

except KeyboardInterrupt:
    print("\n\nSistem monitoring dimatikan.")

Memuat model SVM dan scaler...

SISTEM PENDETEKSI KEBISINGAN RUANGAN SEDANG AKTIF
Mikrofon sedang mendengarkan area sekitar...
Tekan Ctrl+C di terminal untuk mematikan sistem.

[STATUS] Ruangan sangat sunyi...            

Sistem monitoring dimatikan.
